In [1]:
# https://devocean.sk.com/experts/techBoardDetail.do?ID=165806&boardType=experts&page=&searchData=&subIndex=&idList=&searchText=&techType=&searchDataSub=&searchDataMain=&writerID=automan&comment=

In [2]:
import torch
from datasets import Dataset, load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    pipeline,
    TrainingArguments,
)
from peft import LoraConfig, PeftModel
from trl import SFTTrainer

/home/dev/workspace/training-examples/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
from datasets import load_dataset

dataset = load_dataset("beomi/KoAlpaca-v1.1a")
dataset

DatasetDict({
    train: Dataset({
        features: ['instruction', 'output', 'url'],
        num_rows: 21155
    })
})

In [10]:
# BASE_MODEL = "google/gemma-2b"
BASE_MODEL = "beomi/gemma-ko-2b"

model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, device_map={"": 0})
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

Loading checkpoint shards: 100%|██████████| 2/2 [00:00<00:00,  2.74it/s]


In [11]:
prompt = "한국의 트로트라는 음악에 대해 알려줘"
pipe = pipeline("text-generation", model=model, tokenizer=tokenizer, max_new_tokens=256)
outputs = pipe(
    prompt,
    do_sample=True,
    temperature=0.2,
    top_k=50,
    top_p=0.95,
    repetition_penalty=1.2,
    add_special_tokens=True,
)
print(outputs[0]["generated_text"][len(prompt) :])

Device set to use cuda:0


요. 1960년대부터 시작된 우리나라의 대중음악은 서양음악과는 달리 국내에서도 자유롭게 발전할 수 있었어요. 그래서 우리가 아주 좋아하는 노래들이 생겨났죠! 이렇듯, 한국의 대중음악이 세계적인 명성을 얻기까지 많은 사람들의 노력이 있었다고 생각해요. 지금도 계속해서 새로운 장르와 스타일의 음악들을 만들어 내며, 우리에게 사랑받아야 할 음악들입니다 :)


In [31]:
def generate_prompt(example):
    prompt = f"### Instruction: {example['instruction']}\n\n### Response: {example['output']}<eos>"
    return prompt

In [32]:
train_data = dataset["train"]
print(generate_prompt(train_data[0]))

### Instruction: 양파는 어떤 식물 부위인가요? 그리고 고구마는 뿌리인가요?

### Response: 양파는 잎이 아닌 식물의 줄기 부분입니다. 고구마는 식물의 뿌리 부분입니다. 

식물의 부위의 구분에 대해 궁금해하는 분이라면 분명 이 질문에 대한 답을 찾고 있을 것입니다. 양파는 잎이 아닌 줄기 부분입니다. 고구마는 다른 질문과 답변에서 언급된 것과 같이 뿌리 부분입니다. 따라서, 양파는 식물의 줄기 부분이 되고, 고구마는 식물의 뿌리 부분입니다.

 덧붙이는 답변: 고구마 줄기도 볶아먹을 수 있나요? 

고구마 줄기도 식용으로 볶아먹을 수 있습니다. 하지만 줄기 뿐만 아니라, 잎, 씨, 뿌리까지 모든 부위가 식용으로 활용되기도 합니다. 다만, 한국에서는 일반적으로 뿌리 부분인 고구마를 주로 먹습니다.<eos>


In [33]:
lora_config = LoraConfig(
    r=6,
    lora_alpha=8,
    lora_dropout=0.05,
    target_modules=[
        "q_proj",
        "o_proj",
        "k_proj",
        "v_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
    task_type="CAUSAL_LM",
)

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.float16
)

In [34]:
BASE_MODEL = "beomi/gemma-ko-2b"

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL, device_map="auto", quantization_config=bnb_config
)
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
tokenizer.padding_side = "right"

Loading checkpoint shards: 100%|██████████| 2/2 [00:07<00:00,  3.65s/it]


In [ ]:
trainer = SFTTrainer(
    model=model,
    train_dataset=train_data,
    args=TrainingArguments(
        max_length=512,
        output_dir="outputs",
        # num_train_epochs = 1,
        # max_steps=3000,
        # warmup_steps=0.03,
        max_steps=100,
        warmup_steps=10,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=4,
        optim="paged_adamw_8bit",
        learning_rate=2e-4,
        fp16=True,
        logging_steps=10,
        push_to_hub=False,
        report_to="none",
    ),
    peft_config=lora_config,
    formatting_func=generate_prompt,
)

ADAPTER_MODEL = "lora_adapter_it"
trainer.model.save_pretrained(ADAPTER_MODEL)

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL, device_map="auto", torch_dtype=torch.float16
)
model = PeftModel.from_pretrained(
    model, ADAPTER_MODEL, device_map="auto", torch_dtype=torch.float16
)
model = model.merge_and_unload()
model.save_pretrained("gemma-ko-2b-it")

TypeError: SFTTrainer.__init__() got an unexpected keyword argument 'max_length'

In [ ]:
BASE_MODEL = "beomi/gemma-ko-2b"
FINETUNE_MODEL = "./gemma-ko-2b-it"

finetune_model = AutoModelForCausalLM.from_pretrained(
    FINETUNE_MODEL, device_map={"": 0}
)
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
pipe_finetuned = pipeline(
    "text-generation", model=finetune_model, tokenizer=tokenizer, max_new_tokens=512
)
prompt = "한국의 트로트라는 음악에 대해 알려줘"
formatted_prompt = f"### Response: {prompt}\n\n### Response:"
outputs = pipe_finetuned(
    formatted_prompt,
    do_sample=True,
    temperature=0.2,
    top_k=50,
    top_p=0.95,
    repetition_penalty=1.2,
    add_special_tokens=True,
)
print(outputs[0]["generated_text"][len(formatted_prompt) :])